# Task 11: 三种结构 ΔΔG 联合

把三个基于真实 PDB 结构的 ΔΔG 拼在一起：

| ddg | Task | 方法 |
|---|---|---|
| `ddg_struct` | 8 | MJ 统计势能 (接触势 + 溶剂化) |
| `ddg_rasp` | 9 | RaSP 深度学习 (3D-CNN cavity) |
| `ddg_foldx` | 10 | FoldX 金标准 |

**基线**: v3 + PCA(50)  
**联合**: v3 + PCA(50) + 3×ddg = 61 维

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")

ID_COLS = ["KEY", "Gene", "reloc_v3"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]
esm_cols = [c for c in feat_cols if c.startswith("esm_")]
STRUCT_NAMES = ["plddt","sasa","rsa","ss_helix","ss_strand","ss_coil","delta_hydrophobicity"]

X_full_arr = df_feat[feat_cols].values.astype(np.float32)
esm_idx = [feat_cols.index(c) for c in esm_cols]
struct_idx = [feat_cols.index(c) for c in STRUCT_NAMES if c in feat_cols]
X_esm = X_full_arr[:, esm_idx]
X_struct = X_full_arr[:, struct_idx]

y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
groups = df_feat["Gene"].values
full_keys = (df_main["Gene"] + "||" + df_main["Mutation_used"]).tolist()

print(f"n={len(y_bin)}, pos={int(y_bin.sum())}, base_rate={y_bin.sum()/len(y_bin):.4f}")

# ===== 加载三种结构 ddg =====
ddg_sources = {}
ddg_files = {
    "ddg_struct": "ddg_struct.csv",
    "ddg_rasp":   "ddg_rasp.csv",
    "ddg_foldx":  "ddg_foldx.csv",
}

for name, fname in ddg_files.items():
    path = BASE_PATH + fname
    df_d = pd.read_csv(path)
    dcol = "ddg_struct" if name == "ddg_struct" else name
    ddg_map = dict(zip(df_d["KEY"], df_d[dcol]))
    arr = np.array([ddg_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
    n_valid = np.isfinite(arr).sum()
    ddg_sources[name] = arr.reshape(-1, 1)
    print(f"  {name:12s}: {n_valid}/{len(arr)} 有效")

print(f"已加载 {len(ddg_sources)} 种结构 ddg")

n=2179, pos=235, base_rate=0.1078
  ddg_struct  : 2168/2179 有效
  ddg_rasp    : 2168/2179 有效
  ddg_foldx   : 2166/2179 有效
已加载 3 种结构 ddg


## 11a. 同折 CV: PCA(50) vs 单独 ddg vs 全部联合

In [2]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
N_COMP = 50

oof = {}
oof["PCA"] = np.zeros(len(y_bin), dtype=np.float32)
for name in ddg_sources:
    oof[f"PCA+{name}"] = np.zeros(len(y_bin), dtype=np.float32)
oof["PCA+ALL3"] = np.zeros(len(y_bin), dtype=np.float32)

for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full_arr, y_bin, groups)):
    y_tr = y_bin[tr_idx]; y_te = y_bin[te_idx]

    imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
    Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
    Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
    pca = PCA(n_components=N_COMP, random_state=42)
    Xe_tr_pca = pca.fit_transform(Xe_tr).astype(np.float32)
    Xe_te_pca = pca.transform(Xe_te).astype(np.float32)

    imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
    Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
    Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)

    X_tr_base = np.hstack([Xe_tr_pca, Xs_tr]).astype(np.float32)
    X_te_base = np.hstack([Xe_te_pca, Xs_te]).astype(np.float32)

    sw = compute_sample_weight("balanced", y_tr)

    def fit_pred(X_tr, X_te):
        m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.5,
                          objective="binary:logistic", eval_metric="aucpr",
                          random_state=42, n_jobs=-1, tree_method="hist")
        m.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        return m.predict_proba(X_te)[:, 1]

    oof["PCA"][te_idx] = fit_pred(X_tr_base, X_te_base)

    for name, ddg_arr in ddg_sources.items():
        imp_d = SimpleImputer(strategy="median")
        d_tr = imp_d.fit_transform(ddg_arr[tr_idx]).astype(np.float32)
        d_te = imp_d.transform(ddg_arr[te_idx]).astype(np.float32)
        oof[f"PCA+{name}"][te_idx] = fit_pred(
            np.hstack([X_tr_base, d_tr]).astype(np.float32),
            np.hstack([X_te_base, d_te]).astype(np.float32))

    all_d_tr = []; all_d_te = []
    for name, ddg_arr in ddg_sources.items():
        imp_d = SimpleImputer(strategy="median")
        all_d_tr.append(imp_d.fit_transform(ddg_arr[tr_idx]).astype(np.float32))
        all_d_te.append(imp_d.transform(ddg_arr[te_idx]).astype(np.float32))
    oof["PCA+ALL3"][te_idx] = fit_pred(
        np.hstack([X_tr_base] + all_d_tr).astype(np.float32),
        np.hstack([X_te_base] + all_d_te).astype(np.float32))

    vals = {k: roc_auc_score(y_te, oof[k][te_idx]) for k in oof}
    print(f"  Fold {fold}: " + "  ".join([f"{k}={v:.4f}" for k,v in vals.items()]))

  Fold 0: PCA=0.6685  PCA+ddg_struct=0.6863  PCA+ddg_rasp=0.7036  PCA+ddg_foldx=0.6873  PCA+ALL3=0.6827
  Fold 1: PCA=0.6246  PCA+ddg_struct=0.6252  PCA+ddg_rasp=0.6318  PCA+ddg_foldx=0.6461  PCA+ALL3=0.6185
  Fold 2: PCA=0.5527  PCA+ddg_struct=0.5352  PCA+ddg_rasp=0.5386  PCA+ddg_foldx=0.5284  PCA+ALL3=0.5548
  Fold 3: PCA=0.6048  PCA+ddg_struct=0.6035  PCA+ddg_rasp=0.5932  PCA+ddg_foldx=0.6116  PCA+ALL3=0.6013
  Fold 4: PCA=0.6204  PCA+ddg_struct=0.5887  PCA+ddg_rasp=0.5979  PCA+ddg_foldx=0.5602  PCA+ALL3=0.6223


## 11b. 汇总

In [3]:
br = float(y_bin.sum() / len(y_bin))
results = {k: {"auroc": roc_auc_score(y_bin, oof[k]), "auprc": average_precision_score(y_bin, oof[k])} for k in oof}
auroc_pca = results["PCA"]["auroc"]

print(f"\n{'='*80}")
print(f"  Task 11: 三种结构 ddG 联合 (n={len(y_bin)}, pos={int(y_bin.sum())}, base_rate={br:.4f})")
print(f"  {'模型':<35s} {'AUROC':>8s} {'AUPRC':>8s} {'ΔAUROC':>12s}")
print(f"  {'-'*65}")
print(f"  {'PCA(50) 基线':<35s} {auroc_pca:>8.4f} {results['PCA']['auprc']:>8.4f} {'baseline':>12s}")

for name in ddg_sources:
    r = results[f"PCA+{name}"]
    d = r["auroc"] - auroc_pca
    m = " +" if d > 0.005 else ("" if d > -0.005 else " -")
    print(f"  {'PCA+'+name:<35s} {r['auroc']:>8.4f} {r['auprc']:>8.4f} {d:>+12.4f}{m}")

r3 = results["PCA+ALL3"]
d3 = r3["auroc"] - auroc_pca
print(f"  {'─'*65}")
print(f"  {'PCA+ALL3 (三种联合)':<35s} {r3['auroc']:>8.4f} {r3['auprc']:>8.4f} {d3:>+12.4f}")
print(f"{'='*80}")

deltas = {n: results[f"PCA+{n}"]["auroc"]-auroc_pca for n in ddg_sources}
print(f"\n三个单独增益之和: {sum(deltas.values()):+.4f}")
print(f"联合增益:          {d3:+.4f}")
print(f"协同/冗余:         {d3 - sum(deltas.values()):+.4f}  (负值=冗余, 正值=协同)")


  Task 11: 三种结构 ddG 联合 (n=2179, pos=235, base_rate=0.1078)
  模型                                     AUROC    AUPRC       ΔAUROC
  -----------------------------------------------------------------
  PCA(50) 基线                            0.6144   0.1558     baseline
  PCA+ddg_struct                        0.6094   0.1639      -0.0050
  PCA+ddg_rasp                          0.6152   0.1610      +0.0007
  PCA+ddg_foldx                         0.6098   0.1553      -0.0046
  ─────────────────────────────────────────────────────────────────
  PCA+ALL3 (三种联合)                       0.6155   0.1694      +0.0011

三个单独增益之和: -0.0089
联合增益:          +0.0011
协同/冗余:         +0.0100  (负值=冗余, 正值=协同)


## 11c. 特征重要性 (全部联合)

In [4]:
print("\n--- 特征重要性 PCA(50)+ALL3 (fit on all data, 仅用于排名) ---")

imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
Xe_all = scl_e.fit_transform(imp_e.fit_transform(X_esm)).astype(np.float32)
pca_all = PCA(n_components=N_COMP, random_state=42)
Xe_pca_all = pca_all.fit_transform(Xe_all).astype(np.float32)

imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
Xs_all = scl_s.fit_transform(imp_s.fit_transform(X_struct)).astype(np.float32)

parts = [Xe_pca_all, Xs_all]
ddg_order = []
for name in ddg_sources:
    imp_d = SimpleImputer(strategy="median")
    d_all = imp_d.fit_transform(ddg_sources[name]).astype(np.float32)
    parts.append(d_all)
    ddg_order.append(name)

X_all = np.hstack(parts).astype(np.float32)
all_names = [f"PC{i+1}" for i in range(N_COMP)] + STRUCT_NAMES + ddg_order

sw_ratio = (y_bin==0).sum() / max((y_bin==1).sum(), 1)
xgb_fi = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.5, scale_pos_weight=sw_ratio,
                       objective="binary:logistic", eval_metric="aucpr",
                       random_state=42, n_jobs=-1, tree_method="hist")
xgb_fi.fit(X_all, y_bin, verbose=False)

imps = xgb_fi.feature_importances_
idx_sorted = np.argsort(imps)[::-1]

print("Top-20:")
for rank, i in enumerate(idx_sorted[:20]):
    val = imps[i]; bar = "\u2588" * int(val * 2000)
    print(f"  {rank+1:2d}. {all_names[i]:30s}  {val:.5f}  {bar}")

print(f"\n三种结构 ddg 排名 ({len(all_names)} 个特征中):")
for name in ddg_order:
    i = all_names.index(name)
    rank = int(np.where(idx_sorted == i)[0][0]) + 1
    print(f"  {name:15s} 排名={rank:4d}/{len(all_names)}  重要性={imps[i]:.5f}")

print(f"\n结构特征排名:")
for name in STRUCT_NAMES:
    i = all_names.index(name)
    rank = int(np.where(idx_sorted == i)[0][0]) + 1
    marker = " \u2605 top-10!" if rank <= 10 else (" \u2605 top-20!" if rank <= 20 else "")
    print(f"  {name:25s} 排名={rank:3d}/{len(all_names)}  重要性={imps[i]:.5f}{marker}")


--- 特征重要性 PCA(50)+ALL3 (fit on all data, 仅用于排名) ---
Top-20:
   1. PC1                             0.02594  ███████████████████████████████████████████████████
   2. ddg_rasp                        0.02471  █████████████████████████████████████████████████
   3. PC36                            0.02327  ██████████████████████████████████████████████
   4. ss_helix                        0.02097  █████████████████████████████████████████
   5. PC50                            0.02095  █████████████████████████████████████████
   6. PC25                            0.02038  ████████████████████████████████████████
   7. plddt                           0.02015  ████████████████████████████████████████
   8. PC33                            0.02003  ████████████████████████████████████████
   9. PC42                            0.01990  ███████████████████████████████████████
  10. ddg_struct                      0.01966  ███████████████████████████████████████
  11. PC12                       